In [2]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
n = 3
qc = QuantumCircuit(n + 1, n)
qc.x(n)
qc.h(n)
for i in range(n):
    qc.h(i)

for i in range(n):
    qc.h(i)

qc.measure(range(n), range(n))

print(qc)
simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()

counts = result.get_counts()

print("Measurement results:", counts) 

     ┌───┐┌───┐┌─┐      
q_0: ┤ H ├┤ H ├┤M├──────
     ├───┤├───┤└╥┘┌─┐   
q_1: ┤ H ├┤ H ├─╫─┤M├───
     ├───┤├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├┤ H ├─╫──╫─┤M├
     ├───┤├───┤ ║  ║ └╥┘
q_3: ┤ X ├┤ H ├─╫──╫──╫─
     └───┘└───┘ ║  ║  ║ 
c: 3/═══════════╩══╩══╩═
                0  1  2 
Measurement results: {'000': 1024}


In [4]:
#medium
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
n = 3
qc = QuantumCircuit(n + 1, n)
qc.x(n)
qc.h(n)
for i in range(n):
    qc.h(i)
qc.cx(0, n)
qc.cx(1, n)
qc.cx(2, n)
for i in range(n):
    qc.h(i)
qc.measure(range(n), range(n))

print(qc)
simulator = AerSimulator()
compiled = transpile(qc, simulator)
result = simulator.run(compiled, shots=1024).result()
counts = result.get_counts()
print("Measurement results:", counts)

     ┌───┐          ┌───┐     ┌─┐           
q_0: ┤ H ├───────■──┤ H ├─────┤M├───────────
     ├───┤       │  └───┘┌───┐└╥┘     ┌─┐   
q_1: ┤ H ├───────┼────■──┤ H ├─╫──────┤M├───
     ├───┤       │    │  └───┘ ║ ┌───┐└╥┘┌─┐
q_2: ┤ H ├───────┼────┼────■───╫─┤ H ├─╫─┤M├
     ├───┤┌───┐┌─┴─┐┌─┴─┐┌─┴─┐ ║ └───┘ ║ └╥┘
q_3: ┤ X ├┤ H ├┤ X ├┤ X ├┤ X ├─╫───────╫──╫─
     └───┘└───┘└───┘└───┘└───┘ ║       ║  ║ 
c: 3/══════════════════════════╩═══════╩══╩═
                               0       1  2 
Measurement results: {'111': 1024}


In [6]:
#hard
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator


def deutsch_jozsa(n, function_type):
    # n input qubits + 1 output qubit
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(range(n + 1))

    if function_type == "constant":
        # f(x) = 0
        pass

    elif function_type == "balanced":
        # Parity function
        for i in range(n):
            qc.cx(i, n)
    qc.h(range(n))
    qc.measure(range(n), range(n))

    return qc
simulator = AerSimulator()

for n in range(2, 6):

    print("\n==============================")
    print("n =", n)

    for function_type in ["constant", "balanced"]:

        qc = deutsch_jozsa(n, function_type)

        compiled = transpile(qc, simulator)
        result = simulator.run(compiled, shots=1024).result()

        counts = result.get_counts()

        print(function_type.capitalize(), "oracle:")
        print(counts)


n = 2
Constant oracle:
{'00': 1024}
Balanced oracle:
{'11': 1024}

n = 3
Constant oracle:
{'000': 1024}
Balanced oracle:
{'111': 1024}

n = 4
Constant oracle:
{'0000': 1024}
Balanced oracle:
{'1111': 1024}

n = 5
Constant oracle:
{'00000': 1024}
Balanced oracle:
{'11111': 1024}


In [13]:
#challemge
import random
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
def random_balanced_function(n):
    total_inputs = 2 ** n
    half = total_inputs // 2

    values = [0] * half + [1] * half
    random.shuffle(values)

    return values
def create_oracle(n, function_values):
    qc = QuantumCircuit(n + 1)

    output = n

    for x, value in enumerate(function_values):

        if value == 1:

            binary = format(x, f'0{n}b')
            for i, bit in enumerate(binary):
                if bit == '0':
                    qc.x(i)
            qc.mcx(list(range(n)), output)
            for i, bit in enumerate(binary):
                if bit == '0':
                    qc.x(i)

    return qc
def deutsch_jozsa_random(n):
    function_values = random_balanced_function(n)
    qc = QuantumCircuit(n + 1, n)
    qc.x(n)
    qc.h(n)
    qc.h(range(n))
    oracle = create_oracle(n, function_values)
    qc.compose(oracle, inplace=True)
    qc.h(range(n))
    qc.measure(range(n), range(n))
    return qc, function_values
simulator = AerSimulator()
for n in range(2, 6):

    print("\n==============================")
    print("Testing n =", n)
    print("==============================")

    qc, function_values = deutsch_jozsa_random(n)

    compiled = transpile(qc, simulator)

    result = simulator.run(
        compiled,
        shots=1024
    ).result()

    counts = result.get_counts()

    print("Random balanced function:")
    print(function_values)

    print("\nMeasurement result:")
    print(counts)
    zero_state = "0" * n

    if zero_state in counts:
        print("\nClassification: Constant")
    else:
        print("\nClassification: Balanced")


Testing n = 2
Random balanced function:
[0, 1, 1, 0]

Measurement result:
{'11': 1024}

Classification: Balanced

Testing n = 3
Random balanced function:
[0, 0, 1, 1, 0, 0, 1, 1]

Measurement result:
{'010': 1024}

Classification: Balanced

Testing n = 4
Random balanced function:
[0, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1]

Measurement result:
{'1100': 565, '1010': 57, '0001': 82, '1001': 69, '1111': 62, '0111': 64, '0100': 66, '0010': 59}

Classification: Balanced

Testing n = 5
Random balanced function:
[1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0]

Measurement result:
{'00001': 135, '01011': 69, '00010': 65, '10111': 18, '00100': 70, '10101': 146, '00101': 22, '10001': 14, '01110': 12, '10011': 141, '00110': 61, '01000': 13, '10010': 66, '01010': 14, '00111': 14, '11001': 65, '11110': 18, '01100': 20, '11100': 14, '11000': 17, '00011': 13, '11010': 17}

Classification: Balanced
